# Run mutant tests using defect4j in pods

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "notebooks"))

from defect4j.podman_defects4j import PodmanD4J, CheckoutHandle, TestRunResult

# Create client (change container name if needed)
client = PodmanD4J(container="defects4j-container")

print("Client created. Container:", "defects4j-container")
print("Available projects (sample):", client.pids()[:20])

Client created. Container: defects4j-container
Available projects (sample): ['Chart', 'Cli', 'Closure', 'Codec', 'Collections', 'Compress', 'Csv', 'Gson', 'JacksonCore', 'JacksonDatabind', 'JacksonXml', 'Jsoup', 'JxPath', 'Lang', 'Math', 'Mockito', 'Time']


In [5]:
project_id = "Mockito"
bug_id = 1
raw = client._exec(f"defects4j info -p {project_id} -b {bug_id}")["stdout"]
print(raw)

proj = client.project_info(project_id)
bug = client.bug_info(project_id, bug_id)

print(bug)

Summary of configuration for Project: Mockito
--------------------------------------------------------------------------------
    Script dir: /defects4j/framework
      Base dir: /defects4j
    Major root: /defects4j/major
      Repo dir: /defects4j/project_repos
--------------------------------------------------------------------------------
    Project ID: Mockito
       Program: mockito
    Build file: /defects4j/framework/projects/Mockito/Mockito.build.xml
--------------------------------------------------------------------------------
           Vcs: Vcs::Git
    Repository: /defects4j/project_repos/mockito.git
     Commit db: /defects4j/framework/projects/Mockito/active-bugs.csv
Number of bugs: 38
--------------------------------------------------------------------------------

Summary for Bug: Mockito-1
--------------------------------------------------------------------------------
Revision ID (fixed version):
4e9d7607825c3c668fd43f19507bfead566c528c
--------------------------

In [3]:
# Checkout buggy (1b) and fixed (1f) in container
bug_checkout: CheckoutHandle = client.checkout("Lang", 1, "b")
fixed_checkout: CheckoutHandle = client.checkout("Lang", 1, "f")
print("Buggy container workdir:", bug_checkout.container_workdir)
print("Fixed container workdir:", fixed_checkout.container_workdir)

KeyboardInterrupt: 

In [ ]:
# Compile (if needed) and run relevant tests (fast)
print("Compiling patched checkout (in container):")
print(client.compile(fixed_checkout)["stdout"][:400])

print("Running relevant tests on buggy checkout:")
before = client.test(bug_checkout, relevant=True)
print("Failing tests (buggy):", before.failing_tests)

print("Running relevant tests on fixed checkout:")
after = client.test(fixed_checkout, relevant=True)
print("Failing tests (fixed):", after.failing_tests)

In [ ]:
# Upload a single mutant file and test it (container-only)
from pathlib import Path

# LOCAL_MUTANT_FILE: path on your host to the mutated .java file
LOCAL_MUTANT_FILE = "/path/to/your/SomeClass.java"   # CHANGE
# CONTAINER_DEST: destination path inside the checked-out project in the container
CONTAINER_DEST = f"{fixed_checkout.container_workdir}/src/main/java/org/your/package/SomeClass.java"

# Backup existing file in container
client.backup_file_in_container(CONTAINER_DEST)

# Upload mutant file (podman cp host -> container)
print("Uploading mutant file to container...")
client.upload_file(LOCAL_MUTANT_FILE, CONTAINER_DEST)

# Compile & test mutated fixed checkout
print("Compile mutated fixed checkout:")
print(client.compile(fixed_checkout)["stdout"][:400])

print("Run tests on mutated fixed version:")
mut_res = client.test(fixed_checkout, relevant=True)
print("Mutant failing tests:", mut_res.failing_tests)
print("Mutant test returncode:", mut_res.returncode)

# Restore original file in container
client.restore_file_in_container(CONTAINER_DEST)
print("Restored original file in container.")